# DM_G5_P0009_nonlinear_grg_kkt_answer

## 0. 정답본 사용 범위

이 정답본은 Phase 9 비선형계획 문제의 formulation, Solver mapping, convexity/concavity 판정, Hessian, KKT, local/global optimum 해석, ML loss landscape 비유를 포함한다. 문제지는 최적점과 목적값을 숨기고, 이 정답본에서만 계산 결과와 판정 이유를 제시한다.

## 1. 공통 판정 루틴

1. decision variables를 잡는다.
2. objective function과 constraints 중 비선형 요소를 찾는다.
3. Solver method를 Simplex LP가 아니라 GRG Nonlinear로 둘지 판단한다.
4. initial values를 바꿔 local optimum 민감성을 확인한다.
5. Hessian 또는 알려진 함수 성질로 objective의 concavity/convexity를 판정한다.
6. feasible region이 convex인지 확인한다.
7. KKT 조건을 필요조건으로 쓰고, convexity/concavity 조건이 있으면 충분조건까지 확장한다.

## 2. 전체 source anchor와 node_id

주요 source anchors: `DM_PDF07:p001:L009`, `DM_PDF07:p002:L002`, `DM_PDF07:p002:L006`, `DM_PDF07:p003:L002`, `DM_PDF07:p003:L006`, `DM_PDF07:p005:L001`, `DM_PDF07:p006:L003`, `DM_PDF07:p007:L003`, `DM_PDF07:p014:L002`, `DM_PDF07:p027:L002`, `DM_PDF07:p028:L002`, `DM_PDF07:p029:L002`, `DM_PDF07:p030:L002`, `DM_PDF07:p031:L002`, `DM_PDF07:p036:L002`, `DM_PDF07:p039:L002`, `DM_PDF07:p043:L002`, `DM_PDF07:p043:L003`, `DM_PDF02:p026:L009`, `DM_PDF05:p027:L002`.

주요 node_id: `n_DM_PDF07.nonlinear_programming`, `n_DM_PDF07.grg_solver`, `n_DM_PDF07.local_global_optimum`, `n_DM_PDF07.initial_solution_sensitivity`, `n_DM_PDF07.convexity_concavity`, `n_DM_PDF07.hessian_test`, `n_DM_PDF07.kkt_conditions`, `n_DM_PDF02.shadow_price`, `n_DM_PDF05.complementary_slackness`.


## 문제 1 정답 - 스마트 온실 투입량 최적화

### 1. Formulation

Decision variables:

\[
X=\text{영양액 투입량},\quad Y=\text{LED 조명시간}
\]

기본 모형:

\[
\max Q(X,Y)=35X^{0.5}Y^{0.4}
\]

subject to

\[
2X+Y\le60,\quad X\ge0,\quad Y\ge0
\]

추가 변형 모형:

\[
\max F(X,Y)=35X^{0.5}Y^{0.4}+14\sin(0.55X)\cos(0.35Y)
\]

### 2. Solver mapping

| Solver 요소 | 대응 |
|---|---|
| changing cells | `X`, `Y` |
| target cell | 기본 모형은 `Q`, 추가 변형은 `F` |
| constraint cells | `2X+Y`, `X`, `Y` |
| RHS cells | `60`, `0`, `0` |
| solving method | GRG Nonlinear |
| initial values | `(1,1)`, `(10,10)`, `(20,5)`, `(3,35)` 등 |

`X^{0.5}Y^{0.4}`는 변수의 거듭제곱과 곱이 결합되어 있으므로 비선형이다. Simplex LP가 아니라 GRG Nonlinear를 선택한다.

### 3. Convexity/concavity와 전체최적성

기본 모형의 가능영역은 선형 부등식과 비음조건의 교집합이므로 convex feasible region이다.

\[
f(X,Y)=35X^aY^b,\quad a=0.5,\ b=0.4
\]

`a>0`, `b>0`, `a+b=0.9<1`인 Cobb-Douglas 함수는 양의 영역에서 concave다. Hessian 관점에서도

\[
f_{XX}<0,\quad f_{YY}<0,\quad \det(H)=35^2ab(1-a-b)X^{2a-2}Y^{2b-2}>0
\]

이므로 Hessian은 negative definite이다. 최대화 문제에서 concave objective와 convex feasible region이므로 KKT를 만족하는 해는 전체최적해로 보장된다.

KKT에서 내부 양수해를 가정하면 예산 제약은 binding이다.

\[
\mathcal L=35X^{0.5}Y^{0.4}-\lambda(2X+Y-60)
\]

\[
0.5\cdot35X^{-0.5}Y^{0.4}-2\lambda=0
\]

\[
0.4\cdot35X^{0.5}Y^{-0.6}-\lambda=0
\]

두 stationarity 식의 비율에서 `Y=1.6X`, 그리고 `2X+Y=60`이므로

\[
X^*=16.667,\quad Y^*=26.667,\quad Q^*\approx531.349
\]

### 4. 추가 변형 모형과 multi-start

`14sin(0.55X)cos(0.35Y)`는 주기적 요동을 만들기 때문에 Hessian 부호가 위치마다 바뀔 수 있다. 이 경우 contour에 여러 봉우리가 생기며, GRG는 시작점에 따라 다른 local optimum에 도달할 수 있다. 격자 진단 기준으로는 대략 `(X,Y)=(17.75,24.50)` 부근에서 큰 값이 보이지만, 이것은 시각적/수치적 진단이지 수학적 전체최적 증명이 아니다.

오답 진단: “Solver가 해를 하나 보여줬으므로 추가 변형 모형의 전체 최적해가 증명되었다”는 틀렸다. GRG 결과는 local optimum 후보로 보아야 하며, nonconvex landscape에서는 multi-start, contour 검토, 전역 탐색 또는 별도 증명이 필요하다.

ML analogy: 기본 모형은 concave reward landscape라 local maximum이 global maximum으로 연결된다. 추가 변형 모형은 요동이 있는 nonconvex landscape라 initialization에 따라 다른 봉우리로 갈 수 있다.

source anchors: `DM_PDF07:p001:L009`, `DM_PDF07:p002:L006`, `DM_PDF07:p003:L002`, `DM_PDF07:p003:L006`, `DM_PDF07:p006:L003`, `DM_PDF07:p007:L003`.

node_id: `n_DM_PDF07.nonlinear_programming`, `n_DM_PDF07.grg_solver`, `n_DM_PDF07.local_global_optimum`, `n_DM_PDF07.initial_solution_sensitivity`, `n_DM_PDF07.convexity_concavity`.


#### 문항별 시각화 학습자료 - 문제 1

| 항목 | 설계 |
|---|---|
| 시각화 목적 | concave 기본 모형에서는 KKT 해가 왜 전체최적해가 되는지, nonconvex 변형에서는 왜 multi-start가 필요한지 확인한다. |
| 사용할 데이터 | 문제의 `Q(X,Y)`, `F(X,Y)`, `2X+Y<=60`, 네 개 초기해 후보 |
| 필요한 전처리 | feasible mask 생성, Cobb-Douglas Hessian, `F` gradient, feasible set projection |
| 코드 셀 설계 | contour, KKT normal/gradient arrow, Hessian directional curvature, multi-start ascent trajectory |
| 그래프 해석 포인트 | `∇Q = lambda*(2,1)`이면 binding boundary에서 접선 방향 개선이 멈춘다. 방향별 2차곡률이 음수이면 봉우리가 하나인 오목 구조다. |
| 학생이 자주 하는 오해 | “GRG가 찍은 점 하나 = 전체최적”이라고 단정한다. nonconvex 변형에서는 local peak가 여러 개일 수 있다. |
| 체크포인트 질문 | 기본 `Q`와 변형 `F` 중 어느 쪽에서 KKT가 전체최적성까지 보장되는가? 그 이유는 objective와 feasible region 중 어느 조건 때문인가? |

관찰:

원인:

제한:

결론:


In [ ]:
# 문제 1 정답 시각화: KKT normal, Hessian curvature, multi-start trajectory
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 30, 601)
y = np.linspace(0, 60, 1201)
X, Y = np.meshgrid(x, y)
feasible = (2*X + Y <= 60)
Q = 35*np.sqrt(X)*np.power(Y, 0.4)
F = Q + 14*np.sin(0.55*X)*np.cos(0.35*Y)
Qm = np.where(feasible, Q, np.nan)
Fm = np.where(feasible, F, np.nan)

x_star = 16.6666666667
y_star = 26.6666666667
q_star = 35*np.sqrt(x_star)*(y_star**0.4)

def grad_Q(p):
    xv, yv = np.maximum(p, 1e-9)
    return np.array([
        35*0.5*(xv**-0.5)*(yv**0.4),
        35*0.4*(xv**0.5)*(yv**-0.6),
    ])

def hessian_Q(xv, yv):
    fxx = 35*0.5*(-0.5)*(xv**(-1.5))*(yv**0.4)
    fyy = 35*0.4*(-0.6)*(xv**0.5)*(yv**(-1.6))
    fxy = 35*0.5*0.4*(xv**(-0.5))*(yv**(-0.6))
    return np.array([[fxx, fxy], [fxy, fyy]])

def grad_F(p):
    xv, yv = np.maximum(p, 1e-9)
    gq = grad_Q(np.array([xv, yv]))
    return gq + np.array([
        14*0.55*np.cos(0.55*xv)*np.cos(0.35*yv),
        -14*0.35*np.sin(0.55*xv)*np.sin(0.35*yv),
    ])

def project_greenhouse(p):
    p = np.array(p, dtype=float)
    p = np.maximum(p, 0)
    if 2*p[0] + p[1] > 60:
        excess = (2*p[0] + p[1] - 60) / 5.0
        p = p - excess*np.array([2.0, 1.0])
        if p[0] < 0:
            p = np.array([0.0, min(max(p[1], 0), 60.0)])
        if p[1] < 0:
            p = np.array([min(max(p[0], 0), 30.0), 0.0])
    return p

def ascend(start, grad, step=0.06, n=180):
    p = project_greenhouse(start)
    path = [p.copy()]
    for k in range(n):
        g = grad(p)
        p = project_greenhouse(p + step/(1 + 0.01*k)*g)
        path.append(p.copy())
    return np.array(path)

starts = np.array([[1, 1], [10, 10], [20, 5], [3, 35]], dtype=float)
paths = [ascend(s, grad_F) for s in starts]
end_values = [35*np.sqrt(p[-1,0])*(p[-1,1]**0.4) + 14*np.sin(0.55*p[-1,0])*np.cos(0.35*p[-1,1]) for p in paths]
idx = np.nanargmax(Fm)
iy, ix = np.unravel_index(idx, Fm.shape)
f_x, f_y, f_val = X[iy, ix], Y[iy, ix], Fm[iy, ix]

g_star = grad_Q(np.array([x_star, y_star]))
lambda_star = g_star[1]
normal = np.array([2.0, 1.0])
scaled_grad = g_star / np.linalg.norm(g_star) * 5
scaled_normal = normal / np.linalg.norm(normal) * 5

H_star = hessian_Q(x_star, y_star)
angles = np.linspace(0, 2*np.pi, 360)
dirs = np.c_[np.cos(angles), np.sin(angles)]
curv = np.einsum('bi,ij,bj->b', dirs, H_star, dirs)

fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), constrained_layout=True)
ax = axes[0, 0]
cs = ax.contourf(X, Y, Qm, levels=32, cmap='viridis')
ax.plot(x, 60 - 2*x, color='white', linewidth=2, label='2X+Y=60')
ax.scatter([x_star], [y_star], color='white', edgecolor='black', s=105, zorder=6, label='KKT/global optimum')
ax.arrow(x_star, y_star, scaled_grad[0], scaled_grad[1], color='red', head_width=1.0, length_includes_head=True, label='grad Q')
ax.arrow(x_star, y_star, scaled_normal[0], scaled_normal[1], color='cyan', head_width=1.0, length_includes_head=True, label='constraint normal')
ax.set_xlim(0, 30); ax.set_ylim(0, 60)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('기본 Q: grad Q와 lambda*(2,1)의 정렬')
ax.legend(loc='upper right')
fig.colorbar(cs, ax=ax, shrink=0.82)

ax = axes[0, 1]
ax.plot(np.degrees(angles), curv, color='navy')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('방향 각도 theta')
ax.set_ylabel('v.T @ H @ v')
ax.set_title('Hessian directional curvature at optimum: 모두 음수')
ax.text(0.03, 0.08, f'eigenvalues={np.round(np.linalg.eigvals(H_star), 4)}', transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.9))

ax = axes[1, 0]
cs = ax.contourf(X, Y, Fm, levels=34, cmap='plasma')
ax.plot(x, 60 - 2*x, color='white', linewidth=2, label='2X+Y=60')
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for s, path, color in zip(starts, paths, colors):
    ax.plot(path[:,0], path[:,1], color=color, linewidth=2)
    ax.scatter([s[0]], [s[1]], color=color, edgecolor='white', s=45)
    ax.scatter([path[-1,0]], [path[-1,1]], color=color, edgecolor='black', s=70)
ax.scatter([f_x], [f_y], color='white', edgecolor='black', s=105, zorder=7, label='grid global 후보')
ax.set_xlim(0, 30); ax.set_ylim(0, 60)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('추가 F: 시작점별 projected gradient ascent 경로')
ax.legend(loc='upper right')
fig.colorbar(cs, ax=ax, shrink=0.82)

ax = axes[1, 1]
labels = [f'({int(s[0])},{int(s[1])})' for s in starts]
ax.bar(labels, end_values, color=colors)
ax.axhline(f_val, color='black', linestyle='--', linewidth=1.5, label='grid global 후보값')
ax.set_ylabel('도착점 F 값')
ax.set_title('multi-start 도착값 비교: local/global 구분 훈련')
ax.legend()
plt.show()

print(f'기본 모형 전체최적: X={x_star:.3f}, Y={y_star:.3f}, Q={q_star:.3f}')
print(f'KKT lambda={lambda_star:.4f}, grad_Q={np.round(g_star,4)}, lambda*(2,1)={np.round(lambda_star*normal,4)}')
print(f'추가 변형 모형 격자 후보: X={f_x:.2f}, Y={f_y:.2f}, F={f_val:.3f}')
for label, path, value in zip(labels, paths, end_values):
    print(f'초기해 {label} -> 도착점 ({path[-1,0]:.2f}, {path[-1,1]:.2f}), F={value:.3f}')


## 문제 2 정답 - 구독형 앱 가격결정

### 1. Formulation

Decision variables:

\[
p_1=\text{상품 1 가격},\quad p_2=\text{상품 2 가격}
\]

Demand functions:

\[
d_1=140-4p_1+1.5p_2,\quad d_2=120+p_1-3p_2
\]

Profit:

\[
Profit=(p_1-8)d_1+(p_2-10)d_2
\]

전개하면

\[
Profit=-4p_1^2-3p_2^2+2.5p_1p_2+162p_1+138p_2-2320
\]

subject to

\[
d_1+d_2\le180
\]

\[
15\le p_1\le50,\quad 12\le p_2\le45
\]

\[
d_1\ge0,\quad d_2\ge0
\]

수요함수 자체는 선형이지만 이익은 `(가격-원가)×수요`라서 `p_1^2`, `p_2^2`, `p_1p_2`가 생긴다. 따라서 전체 문제는 선형계획이 아니라 비선형계획이다.

### 2. Solver mapping

| Solver 요소 | 대응 |
|---|---|
| changing cells | `p1`, `p2` |
| target cell | `Profit` 최대화 셀 |
| constraint cells | `d1+d2`, `p1`, `p2`, `d1`, `d2` |
| RHS cells | `180`, 가격 하한/상한, `0` |
| solving method | GRG Nonlinear |
| initial values | 예: `(25,25)`, `(40,20)`, `(20,40)` 등 |

### 3. Hessian과 전체최적성

\[
H=\begin{bmatrix}-8&2.5\\2.5&-6\end{bmatrix}
\]

leading principal minor는 `-8<0`, determinant는

\[
(-8)(-6)-2.5^2=41.75>0
\]

이므로 Hessian은 negative definite이고 Profit은 strictly concave다. 모든 제약식은 선형이므로 feasible region은 convex다. 최대화 문제에서 concave objective와 convex feasible region을 가지므로 KKT 조건을 만족하는 feasible solution은 전체최적해다.

stationarity를 풀면 내부해

\[
p_1^*=31.545,\quad p_2^*=36.144
\]

이고,

\[
d_1=68.036,\quad d_2=43.114,\quad Profit^*\approx2729.054
\]

이다. `d1+d2=111.150`으로 총수요 제약은 slack이 있고, 가격 bounds와 demand nonnegativity도 binding이 아니다.

### 4. 오답 진단

“수요함수가 선형이므로 전체 문제는 선형계획이다”는 틀렸다. LP 여부는 수요함수만 보고 판단하지 않는다. 목적함수와 제약식 전체를 봐야 하며, 여기서는 가격과 수요의 곱으로 2차항이 발생한다.

ML analogy: 이미 주어진 반응식 위에서 가격이라는 decision variable을 최적화하는 문제다. 실제 학습을 새로 하는 문제가 아니라, 주어진 response surface 위에서 최적 의사결정을 찾는 NLP다.

source anchors: `DM_PDF07:p027:L002`, `DM_PDF07:p028:L002`, `DM_PDF07:p029:L002`, `DM_PDF07:p030:L002`, `DM_PDF07:p031:L002`, `DM_PDF07:p014:L002`, `DM_PDF07:p007:L003`.

node_id: `n_DM_PDF07.pricing_model`, `n_DM_PDF07.nonlinear_programming`, `n_DM_PDF07.hessian_test`, `n_DM_PDF07.convexity_concavity`, `n_DM_PDF07.global_optimum_sufficient_conditions`, `n_DM_PDF07.grg_solver`.


#### 문항별 시각화 학습자료 - 문제 2

| 항목 | 설계 |
|---|---|
| 시각화 목적 | 가격결정 문제가 왜 “수요는 선형이지만 이익은 concave quadratic NLP”인지 시각적으로 확인한다. |
| 사용할 데이터 | `p1`, `p2`, `d1`, `d2`, Profit, 선형 제약식 |
| 필요한 전처리 | feasible mask 생성, Hessian 고유값/방향별 곡률 계산, 최적점 slack 계산 |
| 코드 셀 설계 | Profit contour, Hessian curvature plot, constraint slack bar, KKT multiplier status bar |
| 그래프 해석 포인트 | Hessian 방향별 곡률이 모두 음수이면 Profit surface는 봉우리형이다. 최적점이 interior이면 모든 제약 multiplier는 0이고 `grad Profit=0`이 KKT stationarity다. |
| 학생이 자주 하는 오해 | 수요식만 보고 LP라고 판단하거나, feasible region만 convex이면 충분하다고 생각한다. |
| 체크포인트 질문 | 이 문제에서 전체최적성은 “GRG가 찾았기 때문”인가, 아니면 “concave objective + convex feasible region” 때문인가? |

관찰:

원인:

제한:

결론:


In [ ]:
# 문제 2 정답 시각화: concave profit, Hessian, slack, KKT multiplier 상태
import numpy as np
import matplotlib.pyplot as plt

p1 = np.linspace(15, 50, 350)
p2 = np.linspace(12, 45, 350)
P1, P2 = np.meshgrid(p1, p2)
D1 = 140 - 4*P1 + 1.5*P2
D2 = 120 + P1 - 3*P2
Profit = (P1 - 8)*D1 + (P2 - 10)*D2
feasible = (D1 + D2 <= 180) & (D1 >= 0) & (D2 >= 0)
Profit_masked = np.where(feasible, Profit, np.nan)

H = np.array([[-8.0, 2.5], [2.5, -6.0]])
linear = np.array([162.0, 138.0])
p_star = np.linalg.solve(-H, linear)
p1_star, p2_star = p_star
D1_star = 140 - 4*p1_star + 1.5*p2_star
D2_star = 120 + p1_star - 3*p2_star
profit_star = (p1_star - 8)*D1_star + (p2_star - 10)*D2_star

angles = np.linspace(0, 2*np.pi, 360)
dirs = np.c_[np.cos(angles), np.sin(angles)]
curv = np.einsum('bi,ij,bj->b', dirs, H, dirs)

grad_star = H @ p_star + linear
slacks = {
    'd1+d2<=180': 180 - (D1_star + D2_star),
    'p1>=15': p1_star - 15,
    'p1<=50': 50 - p1_star,
    'p2>=12': p2_star - 12,
    'p2<=45': 45 - p2_star,
    'd1>=0': D1_star,
    'd2>=0': D2_star,
}
multipliers = {key: 0.0 for key in slacks}

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
ax = axes[0, 0]
cs = ax.contourf(P1, P2, Profit_masked, levels=34, cmap='magma')
ax.contour(P1, P2, feasible.astype(int), levels=[0.5], colors='white', linewidths=2)
ax.scatter([p1_star], [p2_star], color='white', edgecolor='black', s=105, zorder=5, label='global optimum')
ax.set_xlabel('p1'); ax.set_ylabel('p2')
ax.set_title('Profit contour: strictly concave quadratic')
ax.legend(loc='upper right')
fig.colorbar(cs, ax=ax, label='Profit')

ax = axes[0, 1]
ax.plot(np.degrees(angles), curv, color='darkgreen')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('방향 각도 theta')
ax.set_ylabel('v.T @ H @ v')
ax.set_title('Hessian directional curvature: 전 방향 음수')
ax.text(0.03, 0.08, f'eigenvalues={np.round(np.linalg.eigvals(H), 4)}', transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.9))

ax = axes[1, 0]
labels = list(slacks.keys())
values = list(slacks.values())
ax.barh(labels, values, color='#74A9CF')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('slack at optimum')
ax.set_title('최적점 constraint slack: 모두 양수이면 비활성')

ax = axes[1, 1]
mu_labels = list(multipliers.keys())
mu_values = list(multipliers.values())
ax.barh(mu_labels, mu_values, color='#FDD0A2')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlim(0, 1)
ax.set_xlabel('KKT multiplier')
ax.set_title('Interior optimum: 모든 제약 multiplier = 0')
ax.text(0.05, 0.08, f'||grad Profit||={np.linalg.norm(grad_star):.2e}', transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.9))
plt.show()

print('Hessian =')
print(H)
print(f'determinant={np.linalg.det(H):.3f}, eigenvalues={np.round(np.linalg.eigvals(H), 6)}')
print(f'p1*={p1_star:.3f}, p2*={p2_star:.3f}, d1={D1_star:.3f}, d2={D2_star:.3f}, Profit={profit_star:.3f}')
print(f'grad Profit at optimum={np.round(grad_star, 10)}')
for name, slack in slacks.items():
    print(f'{name}: slack={slack:.3f}, multiplier=0.000, product=0.000')


## 문제 3 정답 - ML 기반 광고예산 반응모형

### 1. Formulation

Decision variables:

\[
x_1=\text{채널 1 광고예산},\quad x_2=\text{채널 2 광고예산}
\]

기본 모형:

\[
\max R(x_1,x_2)=90\ln(1+x_1)+75\ln(1+x_2)-1.8x_1-1.2x_2
\]

subject to

\[
x_1+x_2\le45,\quad x_1\ge0,\quad x_2\ge0
\]

대안 모형:

\[
\max R_{alt}(x_1,x_2)=R(x_1,x_2)+8\sin(0.35x_1)\cos(0.30x_2)
\]

### 2. Gradient와 Hessian

기본 모형의 gradient는

\[
\nabla R=\left(\frac{90}{1+x_1}-1.8,\ \frac{75}{1+x_2}-1.2\right)
\]

Hessian은

\[
H=\begin{bmatrix}
-\frac{90}{(1+x_1)^2}&0\\
0&-\frac{75}{(1+x_2)^2}
\end{bmatrix}
\]

가능영역에서 대각원소가 모두 음수이고 교차항은 0이므로 Hessian은 negative definite다. 따라서 `R`은 strictly concave다. 제약식은 선형이므로 feasible region은 convex다.

### 3. KKT와 전체최적성

최대화 문제로 쓰면

\[
\mathcal L=R-\lambda(x_1+x_2-45)+\mu_1x_1+\mu_2x_2
\]

\[
\lambda\ge0,\quad \mu_1\ge0,\quad \mu_2\ge0
\]

stationarity:

\[
\frac{90}{1+x_1}-1.8-\lambda+\mu_1=0
\]

\[
\frac{75}{1+x_2}-1.2-\lambda+\mu_2=0
\]

primal feasibility:

\[
x_1+x_2\le45,\quad x_1\ge0,\quad x_2\ge0
\]

dual feasibility:

\[
\lambda,\mu_1,\mu_2\ge0
\]

complementary slackness:

\[
\lambda(45-x_1-x_2)=0,\quad \mu_1x_1=0,\quad \mu_2x_2=0
\]

최적해는 내부 양수해이고 예산 제약이 binding이므로 `mu1=mu2=0`, `x1+x2=45`이다. 따라서

\[
\frac{90}{1+x_1}-1.8=\frac{75}{1+x_2}-1.2=\lambda
\]

을 풀면

\[
x_1^*=22.628,\quad x_2^*=22.372,\quad \lambda^*\approx2.009
\]

\[
R^*\approx453.408
\]

이다. `lambda`는 예산 RHS를 작은 범위에서 1단위 늘렸을 때 최대 반응값이 약 `2.009`만큼 증가한다는 shadow price식 한계가치로 해석한다.

기본 모형은 concave objective와 convex feasible region을 가지므로 KKT가 전체최적성까지 보장한다. 반대로 대안 모형은 sine-cosine 항 때문에 nonconvex landscape가 될 수 있어 KKT 만족은 local optimum 후보라는 뜻에 머문다.

### 4. 오답 진단

“KKT 조건을 만족하면 항상 전체최적해다”는 틀렸다. 일반 NLP에서 KKT는 필요조건이다. 전체최적성을 주장하려면 objective와 feasible region의 convexity/concavity 구조를 추가로 확인해야 한다.

ML analogy: `R`을 reward maximization landscape로 보면 기본 모형은 단일 봉우리 구조에 가깝다. `R_alt`는 nonconvex neural network loss처럼 여러 local basin 또는 봉우리를 만들 수 있으며, initialization이 결과에 영향을 줄 수 있다. 여기서는 surrogate가 이미 주어졌고, 우리는 그 위에서 decision variable을 최적화한다.

source anchors: `DM_PDF07:p003:L002`, `DM_PDF07:p004:L002`, `DM_PDF07:p005:L001`, `DM_PDF07:p008:L003`, `DM_PDF07:p009:L002`, `DM_PDF07:p010:L003`, `DM_PDF07:p011:L002`, `DM_PDF07:p036:L002`, `DM_PDF07:p039:L002`, `DM_PDF07:p043:L002`, `DM_PDF07:p043:L003`, `DM_PDF02:p026:L009`, `DM_PDF05:p027:L002`.

node_id: `n_DM_PDF07.kkt_conditions`, `n_DM_PDF07.local_global_optimum`, `n_DM_PDF07.initial_solution_sensitivity`, `n_DM_PDF07.convex_feasible_region`, `n_DM_PDF07.hessian_test`, `n_DM_PDF07.global_optimum_sufficient_conditions`, `n_DM_PDF02.shadow_price`, `n_DM_PDF05.complementary_slackness`.


#### 문항별 시각화 학습자료 - 문제 3

| 항목 | 설계 |
|---|---|
| 시각화 목적 | KKT의 네 조건을 그림으로 분해하고, `lambda`가 예산 RHS의 shadow price로 해석되는 이유를 확인한다. |
| 사용할 데이터 | `R(x1,x2)`, `R_alt(x1,x2)`, `x1+x2<=45`, lower bounds, multi-start 후보 |
| 필요한 전처리 | KKT 해 계산, Hessian 부호 확인, budget RHS 변화별 최적값 계산, projected gradient ascent |
| 코드 셀 설계 | KKT normal/gradient contour, complementary slackness bar, shadow price tangent plot, nonconvex multi-start trajectory |
| 그래프 해석 포인트 | binding 예산선에서는 `grad R = lambda*(1,1)`이다. `lambda>0`이고 budget slack이 0이면 예산 제약이 실제로 최적값을 제한한다. |
| 학생이 자주 하는 오해 | KKT 만족을 항상 전체최적성으로 해석하거나, multiplier를 artificial variable처럼 임시 변수로 착각한다. |
| 체크포인트 질문 | `lambda`가 양수일 때 RHS를 조금 늘리면 objective가 어느 방향으로 얼마나 변하는가? `R_alt`에서도 같은 보장이 가능한가? |

관찰:

원인:

제한:

결론:


In [ ]:
# 문제 3 정답 시각화: KKT normal, complementary slackness, shadow price, nonconvex multi-start
import numpy as np
import matplotlib.pyplot as plt

x1 = np.linspace(0, 45, 901)
x2 = np.linspace(0, 45, 901)
X1, X2 = np.meshgrid(x1, x2)
feasible = (X1 + X2 <= 45)
R = 90*np.log1p(X1) + 75*np.log1p(X2) - 1.8*X1 - 1.2*X2
R_alt = R + 8*np.sin(0.35*X1)*np.cos(0.30*X2)
Rm = np.where(feasible, R, np.nan)
Ram = np.where(feasible, R_alt, np.nan)

def solve_budget(B):
    lo, hi = 1e-9, B + 2 - 1e-9
    def balance(a):
        b = B + 2 - a
        return 90/a - 75/b - 0.6
    for _ in range(140):
        mid = (lo + hi) / 2
        if balance(lo) * balance(mid) <= 0:
            hi = mid
        else:
            lo = mid
    a = (lo + hi) / 2
    b = B + 2 - a
    xv, yv = a - 1, b - 1
    lam = 90/(1 + xv) - 1.8
    value = 90*np.log1p(xv) + 75*np.log1p(yv) - 1.8*xv - 1.2*yv
    return xv, yv, lam, value

x1_star, x2_star, lam_star, R_star = solve_budget(45)
grad_star = np.array([90/(1+x1_star)-1.8, 75/(1+x2_star)-1.2])
normal = np.array([1.0, 1.0])

budgets = np.linspace(38, 52, 29)
solutions = np.array([solve_budget(B) for B in budgets])
values_B = solutions[:, 3]
tangent = R_star + lam_star*(budgets - 45)

# R_alt multi-start projected gradient ascent
starts = np.array([[0, 0], [5, 5], [15, 10], [10, 30], [30, 10], [22, 22]], dtype=float)

def grad_R_alt(p):
    xv, yv = np.maximum(p, 1e-9)
    return np.array([
        90/(1+xv) - 1.8 + 8*0.35*np.cos(0.35*xv)*np.cos(0.30*yv),
        75/(1+yv) - 1.2 - 8*0.30*np.sin(0.35*xv)*np.sin(0.30*yv),
    ])

def project_budget(p):
    p = np.maximum(np.array(p, dtype=float), 0)
    if p.sum() > 45:
        excess = (p.sum() - 45) / 2
        p = p - excess
        if p[0] < 0:
            p = np.array([0.0, 45.0])
        if p[1] < 0:
            p = np.array([45.0, 0.0])
    return p

def ascend(start, step=0.07, n=220):
    p = project_budget(start)
    path = [p.copy()]
    for k in range(n):
        p = project_budget(p + step/(1+0.01*k)*grad_R_alt(p))
        path.append(p.copy())
    return np.array(path)

paths = [ascend(s) for s in starts]
end_values = [90*np.log1p(p[-1,0]) + 75*np.log1p(p[-1,1]) - 1.8*p[-1,0] - 1.2*p[-1,1] + 8*np.sin(0.35*p[-1,0])*np.cos(0.30*p[-1,1]) for p in paths]
idx = np.nanargmax(Ram)
iy, ix = np.unravel_index(idx, Ram.shape)
alt_x1, alt_x2, alt_val = X1[iy, ix], X2[iy, ix], Ram[iy, ix]

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.8), constrained_layout=True)
ax = axes[0, 0]
cs = ax.contourf(X1, X2, Rm, levels=34, cmap='cividis')
ax.plot(x1, 45 - x1, color='white', linewidth=2, label='x1+x2=45')
ax.scatter([x1_star], [x2_star], color='white', edgecolor='black', s=110, zorder=6, label='KKT/global optimum')
scale = 4.5
ax.arrow(x1_star, x2_star, *(grad_star/np.linalg.norm(grad_star)*scale), color='red', head_width=0.8, length_includes_head=True, label='grad R')
ax.arrow(x1_star, x2_star, *(normal/np.linalg.norm(normal)*scale), color='cyan', head_width=0.8, length_includes_head=True, label='budget normal')
ax.set_xlim(0, 45); ax.set_ylim(0, 45)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('KKT geometry: grad R = lambda*(1,1)')
ax.legend(loc='upper right')
fig.colorbar(cs, ax=ax, shrink=0.82)

ax = axes[0, 1]
cs_labels = ['budget', 'x1 lower', 'x2 lower']
slack_values = [45 - x1_star - x2_star, x1_star, x2_star]
mult_values = [lam_star, 0.0, 0.0]
xpos = np.arange(len(cs_labels))
width = 0.36
ax.bar(xpos - width/2, slack_values, width, label='slack 또는 primal value', color='#9ECAE1')
ax.bar(xpos + width/2, mult_values, width, label='multiplier', color='#FDD0A2')
ax.set_xticks(xpos)
ax.set_xticklabels(cs_labels)
ax.set_title('Complementary slackness: product가 0인지 확인')
ax.legend()
for i, (s, m) in enumerate(zip(slack_values, mult_values)):
    ax.text(i, max(s, m) + 0.7, f'product={s*m:.2e}', ha='center', fontsize=9)

ax = axes[1, 0]
ax.plot(budgets, values_B, marker='o', label='R*(budget RHS)')
ax.plot(budgets, tangent, linestyle='--', color='crimson', label=f'tangent slope lambda={lam_star:.3f}')
ax.axvline(45, color='black', linewidth=1)
ax.set_xlabel('budget RHS')
ax.set_ylabel('optimal R value')
ax.set_title('Shadow price: RHS 변화와 최적값의 기울기')
ax.legend()

ax = axes[1, 1]
cs = ax.contourf(X1, X2, Ram, levels=34, cmap='plasma')
ax.plot(x1, 45 - x1, color='white', linewidth=2, label='x1+x2=45')
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628']
for s, path, color in zip(starts, paths, colors):
    ax.plot(path[:,0], path[:,1], color=color, linewidth=2)
    ax.scatter([s[0]], [s[1]], color=color, edgecolor='white', s=45)
    ax.scatter([path[-1,0]], [path[-1,1]], color=color, edgecolor='black', s=65)
ax.scatter([alt_x1], [alt_x2], color='white', edgecolor='black', s=110, zorder=7, label='grid global 후보')
ax.set_xlim(0, 45); ax.set_ylim(0, 45)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('R_alt: multi-start trajectory와 local peak 위험')
ax.legend(loc='upper right')
fig.colorbar(cs, ax=ax, shrink=0.82)
plt.show()

print(f'기본 모형 KKT/global optimum: x1={x1_star:.3f}, x2={x2_star:.3f}, R={R_star:.3f}, lambda={lam_star:.3f}')
print(f'grad R={np.round(grad_star,4)}, lambda*(1,1)={np.round(lam_star*normal,4)}')
print('Complementary slackness products:', [round(s*m, 10) for s, m in zip(slack_values, mult_values)])
print(f'대안 모형 격자 global 후보: x1={alt_x1:.2f}, x2={alt_x2:.2f}, R_alt={alt_val:.3f}')
for start, path, value in zip(starts, paths, end_values):
    print(f'초기해 ({start[0]:.0f},{start[1]:.0f}) -> 도착점 ({path[-1,0]:.2f},{path[-1,1]:.2f}), R_alt={value:.3f}')


## 채점 기준

| 항목 | 배점 | 확인 기준 |
|---|---:|---|
| nonlinear formulation과 Solver mapping | 20 | decision variable, objective, constraints, GRG method를 정확히 연결 |
| GRG/local/global/initial value 해석 | 20 | local optimum 가능성과 multi-start 필요성을 설명 |
| convexity/concavity와 Hessian 판정 | 25 | Hessian 부호, negative definite, convex feasible region을 정확히 판정 |
| KKT 조건과 sufficient condition 구분 | 25 | primal/stationarity/dual/complementary slackness와 충분조건 구분 |
| ML analogy, visualization, source anchor/node_id | 10 | 비유를 최적화 언어로 제한하고 anchor/node_id 포함 |

## 오답튜터 기준표

| 오답 유형 | 왜 틀렸는가 | 교정 질문 |
|---|---|---|
| 목적함수가 비선형인데 LP로 푸는 오류 | Simplex LP는 선형식 구조를 전제로 한다. | 목적함수와 제약식 중 어디에 곱, 제곱, 로그, 삼각함수가 있는가? |
| 제약식만 보고 선형계획이라고 판단 | objective도 선형이어야 LP다. | 수요함수가 아니라 Profit 전체를 전개하면 어떤 항이 생기는가? |
| GRG 결과를 전체최적해로 단정 | GRG는 보통 local optimum 후보를 찾는다. | convexity/concavity 또는 multi-start 증거가 있는가? |
| initial value 민감성을 확인하지 않음 | nonconvex에서는 시작점에 따라 다른 해가 나올 수 있다. | 서로 다른 초기해에서 같은 basin으로 가는가? |
| KKT 필요조건을 충분조건으로 오해 | 일반 NLP에서 KKT는 지역 최적 후보 조건이다. | 목적함수와 가능영역 조건이 충분조건을 만족하는가? |
| 최대화 문제에서 convex objective를 보장 조건으로 착각 | 최대화의 충분조건은 concave objective다. | max인지 min인지 먼저 확인했는가? |
| 최소화 문제에서 concave objective를 보장 조건으로 착각 | 최소화의 충분조건은 convex objective다. | Hessian 부호와 문제 방향이 맞는가? |
| Hessian 부호 판정을 반대로 함 | negative definite는 concave, positive definite는 convex다. | leading principal minors와 determinant를 확인했는가? |
| feasible region convexity와 objective convexity를 혼동 | 둘은 별도 조건이다. | 가능영역 조건과 목적함수 조건을 따로 썼는가? |
| Lagrange multiplier와 artificial variable 혼동 | multiplier는 제약 한계가치, artificial variable은 simplex 초기 BFS용 임시 변수다. | 최종해에서 0이어야 하는 보조변수와 제약의 한계가치를 구분했는가? |
| ML analogy를 실제 학습/평가 문제로 바꿈 | 여기서는 surrogate가 주어진 뒤 decision variable을 최적화한다. | 학습 문제가 아니라 의사결정 최적화 문제로 유지했는가? |

## 다음 통합 Phase 연결

LP는 선형 구조라 꼭짓점과 Simplex로 풀고, IP는 정수성 때문에 branch-and-bound로 풀며, NLP는 곡면 구조 때문에 GRG, KKT, convexity/concavity로 해석한다. 통합 문제에서는 먼저 문제를 LP/IP/network/NLP로 분류한 뒤, Solver method와 최적성 보장 논리를 맞춰야 한다.
